# RT in-context transfer (choice + RT in the context)

**Exactly the same procedure as the choice-only in-context fine-tune** we already ran —
only the data has RT. Continue training `mpop_rt` on `[task-A + task-B]` sequences
(loss on B's responses), but now every trial carries `<<choice>> <<rtN>>`, so the model
conditions on a person's prior-task **choices AND reaction times** and predicts both.

The shuffled control (own-A vs a stranger's-A) tests person-specificity. We then
**decompose** the transfer into choice vs RT with `--rep`.

Compare to the choice-only in-context result (real < shuffled: DF→recent −0.0059,
kirby→discount −0.0118). Does adding RT to the context make it bigger?

In [ ]:
# setup (RT bin data = output_nl_rt; you already have it from the M_pop_RT run)
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.isdir('/content/sro-minitaur-transfer'):
    !cd /content/sro-minitaur-transfer && git pull
else:
    !git clone https://github.com/YifeiCAO/sro-minitaur-transfer.git /content/sro-minitaur-transfer
!pip -q install peft bitsandbytes scikit-learn
RT = '/content/drive/MyDrive/sro_minitaur/output_nl_rt'
if not os.path.isdir(RT + '/complete'):
    !cd /content/drive/MyDrive/sro_minitaur && unzip -oq output_nl_rt.zip
assert os.path.isdir(RT + '/complete'), 'need output_nl_rt on Drive'
print('RT tasks:', len(os.listdir(RT + '/complete')))

In [ ]:
# 1) train the RT in-context model (continues mpop_rt on [A+B]-with-RT). ~hours.
#    A100-40G: batch 2. 96G H100: --batch-size 6 (much faster). No liger on this path.
!cd /content/sro-minitaur-transfer && python scripts/finetune_incontext.py \
    --mpop /content/drive/MyDrive/sro_minitaur/mpop_rt \
    --nl-dir /content/drive/MyDrive/sro_minitaur/output_nl_rt \
    --subset starting_subset --pairs within \
    --out /content/drive/MyDrive/sro_minitaur/mpop_rt_ic \
    --max-seq-len 6144 --batch-size 2 --grad-accum 4

In [ ]:
# 2) MAIN result: within-domain real-vs-shuffled matrix (choice + RT scored together)
!cd /content/sro-minitaur-transfer && python scripts/build_incontext_matrix.py \
    --mpop /content/drive/MyDrive/sro_minitaur/mpop_rt_ic \
    --nl-dir /content/drive/MyDrive/sro_minitaur/output_nl_rt \
    --subset starting_subset --pairs within --K 10 --max-seq-len 6144 --rep both
!cd /content/sro-minitaur-transfer && python scripts/analyze_incontext_stats.py

In [ ]:
# 3) DECOMPOSE: on the two flagship pairs, score choice-only vs RT-only vs both.
#    real < shuffled means own-A predicts own-B better than a stranger's-A.
import subprocess
base = '/content/drive/MyDrive/sro_minitaur'
pairs = [('directed_forgetting', 'recent_probes'), ('kirby', 'discount_titrate')]
for s, t in pairs:
    for rep in ['both', 'choice', 'rt']:
        print(f'\n===== {s} -> {t}   rep={rep} =====')
        subprocess.run(['python', 'scripts/run_incontext.py',
                        '--mpop', f'{base}/mpop_rt_ic', '--nl-dir', f'{base}/output_nl_rt',
                        '--source', s, '--target', t, '--max-seq-len', '6144', '--rep', rep],
                       cwd='/content/sro-minitaur-transfer')

## How to read

**Cell 2 (matrix, rep=both):** within-domain `id_top1` / permutation p. Compare to the
choice-only in-context (discounting id_top1 ~0.28). Higher = adding RT to the context
improves person-specific cross-task transfer.

**Cell 3 (decompose), per pair the JSON shows `mean_real_minus_shuffled` + `p`:**

| rep | what it isolates |
|---|---|
| `both` | choice + RT together |
| `choice` | only the choice residual transfers |
| `rt` | only the RT residual transfers |

- **`rt` real<shuffled significant** → a person's prior-task RT predicts their next-task
  RT better than a stranger's → RT carries person-specific cross-task signal *in context*.
- **`rt` ≈ 0 but `choice` significant** → the in-context transfer rides on choices, not RT.
- **`both` > `choice`** → RT adds on top of choice.